# Copyright
<pre>
Jelen forráskód a Budapesti Műszaki és Gazdaságtudományi Egyetemen
tartott "Deep Learning a gyakorlatban Python és LUA alapon" tantárgy segédanyagaként készült.

A tantárgy honlapja: http://smartlab.tmit.bme.hu/oktatas-deep-learning
Deep Learning kutatás: http://smartlab.tmit.bme.hu/deep-learning

A forráskódot GPLv3 licensz védi. Újrafelhasználás esetén lehetőség szerint kérejük az alábbi szerzőt értesíteni.

2023, 2026 (c) Gyires-Tóth Bálint (toth.b kukac tmit pont bme pont hu), Zainkó Csaba
</pre>

# 1D konvolúció idősor-regresszióra: hőmérséklet-előrejelzés (órai változat)

Ebben a notebookban egy dimenziós konvolúciós hálózattal (`Conv1D`) **24 órával előre jelezzük a hőmérsékletet**. Valós adatokkal dolgozunk: a jenai (Németország) Max Planck Biogeokémiai Intézet meteorológiai állomása 2009 és 2016 között 10 percenként 14 mennyiséget mért (hőmérséklet, légnyomás, páratartalom, szélsebesség stb.).

A notebook lépései:
1. **Bemelegítés:** mit csinál egy 1D konvolúció? Kézzel megadott szűrők egy egyszerű jelen.
2. **Adatok:** letöltés, előkészítés, ablakozás.
3. **Baseline:** egy nagyon egyszerű, „tanulás nélküli” előrejelzés, amihez mérni tudunk.
4. **1D CNN:** a hálózat felépítése, tanítása és kiértékelése.
5. **Belenézünk a hálóba:** mit tanultak az első réteg szűrői?
6. **Feladatok és ranglista-verseny.**

A példakód elsősorban az 1D konvolúcióra koncentrál. A standardizálást azért használjuk, mert a bemeneti csatornák mértékegysége nagyon különböző (°C, mbar, %, m/s).

## 🛑 Hogyan használd ezt a notebookot?

A notebookban **10 közös megálló** van (🗣️). Ezeknél a teljes futtatás (*Runtime → Run all*) **szándékosan megáll** egy hibaüzenettel (`KozosMegallo`). Kétféle megálló van:

* **Tippelős megálló:** a kódcellában egy vagy több `None  # ✍️` értéket kell kitöltened a saját tippeddel. Ha mindet kitöltötted, futtasd újra a cellát: továbbenged.
* **Megbeszélős megálló:** itt meg kell várni a közös megbeszélést. Utána jelöld ki a megálló kódcelláját, és futtasd tovább a notebookot: *Runtime → Run after* (`Ctrl+F10`).

A ✍️ jelű markdown cellákba írhatod a válaszaidat: **dupla kattintással** szerkesztheted őket, a `Shift+Enter` visszaváltja őket formázott nézetre.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import layers

keras.utils.set_random_seed(42)   # reprodukálhatóság


class KozosMegallo(Exception):
    pass

def megallo(n, *tippek):
    # Közös megálló: a "Run all" itt szándékosan megáll.
    # Ha a megállóhoz tippek tartoznak, és mind ki van töltve, a cella továbbenged.
    if len(tippek) == 0:
        raise KozosMegallo(f"🛑 Közös megálló {n}: várd meg a közös megbeszélést! "
                           "Utána jelöld ki ezt a cellát, és futtass tovább: Runtime → Run after (Ctrl+F10).")
    if any(t is None for t in tippek):
        raise KozosMegallo(f"🛑 Közös megálló {n}: írd be a tipped(ek)et a None helyére, és futtasd újra ezt a cellát!")
    print(f"✅ Megálló {n}: a tipped rögzítve, mehetsz tovább!")

## 1. Bemelegítés: mit csinál egy 1D konvolúció?

Az 1D konvolúciós réteg egy rövid **kernelt** (szűrőt) csúsztat végig az **időtengely** mentén, és minden pozícióban súlyozott összeget számol:

$$y[t] = \sum_{k=0}^{K-1} w[k]\cdot x[t+k] + b$$

* A Keras a bemenetet `(batch, időlépések, csatornák)` alakban várja.
* `padding='valid'` (alapértelmezés) esetén a kernel csak ott számol, ahol teljesen a jelen belül van.
* A mélytanulási keretrendszerekben a „konvolúció” valójában **keresztkorreláció**: a kernelt nem tükrözik. (Tanult szűrőknél ennek nincs jelentősége.)

Először nem tanítunk semmit: egy `Conv1D` rétegbe **kézzel írunk be** két klasszikus szűrőt, és megnézzük, mit csinálnak egy 200 mintás zajos jellel, amelyben a 120. mintánál egy hirtelen ugrás van:
* **mozgóátlag** (`[1/9, ..., 1/9]`, $K=9$),
* **differencia** (`[-1, 0, 1]`, $K=3$).

---
## 🗣️ Közös megálló 1 (≈ 8. perc): tippelj!

Mielőtt lefuttatnánk a szűrőket:
1. Hány elemű lesz a két szűrő kimenete (`padding='valid'`, a bemenet 200 hosszú)? Írd be a tippjeidet a lenti kódcellába!
2. Rajzold le egy papírra (vagy írd le szavakkal), milyen lesz a két kimenet! Melyik lesz sima? Melyiken lesz kiugró érték, és hol?

In [ ]:
tipp_hossz_K9 = None  # ✍️ a K=9 mozgóátlag kimenetének hossza
tipp_hossz_K3 = None  # ✍️ a K=3 differencia szűrő kimenetének hossza
megallo(1, tipp_hossz_K9, tipp_hossz_K3)

In [ ]:
# zajos szinusz, a 120. mintától egy hirtelen ugrással
t = np.linspace(0, 4*np.pi, 200)
rng = np.random.default_rng(0)
signal = np.sin(t) + 0.2*rng.standard_normal(len(t))
signal[120:] += 1.5

def conv1d_fixed(x, kernel):
    # egyetlen szűrős Conv1D réteg, amelynek súlyait mi adjuk meg
    kernel = np.asarray(kernel, dtype='float32').ravel()
    conv = layers.Conv1D(filters=1, kernel_size=len(kernel), use_bias=False)
    conv.build((None, None, 1))
    conv.set_weights([kernel.reshape(-1, 1, 1)])     # súlyok alakja: (kernel_size, bemeneti csatornák, szűrők)
    y = conv(x.reshape(1, -1, 1).astype('float32'))  # bemenet alakja: (batch, időlépések, csatornák)
    return keras.ops.convert_to_numpy(y)[0, :, 0]

moving_avg = np.ones(9) / 9
difference = np.array([-1.0, 0.0, 1.0])

y_avg  = conv1d_fixed(signal, moving_avg)
y_diff = conv1d_fixed(signal, difference)

print("bemenet hossza:", len(signal))
print(f"kimenet hossza (K=9): {len(y_avg)}   (tipped: {tipp_hossz_K9})")
print(f"kimenet hossza (K=3): {len(y_diff)}   (tipped: {tipp_hossz_K3})")
print("Egyezik a numpy keresztkorrelációval?", np.allclose(y_avg, np.correlate(signal, moving_avg, mode='valid'), atol=1e-5))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes[0].plot(signal);  axes[0].set_title("Bemenet: zajos szinusz + ugrás")
axes[1].plot(np.arange(4, 4+len(y_avg)), y_avg, color='tab:green');    axes[1].set_title("Mozgóátlag szűrő (K=9): simítás")
axes[2].plot(np.arange(1, 1+len(y_diff)), y_diff, color='tab:red');    axes[2].set_title("Differencia szűrő [-1, 0, 1]: változások kiemelése")
axes[2].set_xlabel("időlépés")
plt.tight_layout()

A differencia szűrő kimenete az ugrásnál ad nagy értéket, de a zajt is felerősíti. A mozgóátlag eltünteti a zaj nagy részét, de az ugrást is elkeni. Egy konvolúciós hálózat **ugyanilyen szűrőket tanul meg magától**, csak éppen olyanokat, amelyek a feladat szempontjából hasznosak.

---
## 🗣️ Közös megálló 2 (≈ 15. perc): kernel-verseny 🏆

**Feladat:** tervezz olyan kernelt, amely az ugrást a lehető legjobban kiemeli a zajból!

**Pontozás:** a `score_kernel` függvény elosztja a kimenet ugrásnál mért csúcsát (abszolút értékben) a kimenet szórásával az ugrástól távol eső részeken. Minél nagyobb, annál jobb. A szűrő hossza 2 és 31 között lehet.

Írd be a kerneledet a `sajat_kernel` változóba (pl. `[-1, 0, 1]`), futtasd, és **írd fel a pontszámodat a táblára!** Annyiszor próbálkozhatsz, ahányszor csak szeretnél. Az alábbi cella megmutatja a két mintaszűrő pontszámát.

In [ ]:
JUMP = 120   # az ugrás helye a bemenetben

def score_kernel(kernel):
    kernel = np.asarray(kernel, dtype='float32').ravel()
    K = len(kernel)
    assert 2 <= K <= 31, "A kernel hossza 2 és 31 között legyen!"
    y = conv1d_fixed(signal, kernel)
    # azok a kimeneti pozíciók, amelyek ablaka a 119. és a 120. mintát is tartalmazza
    jump_zone = np.arange(JUMP + 1 - K, JUMP)
    peak = np.max(np.abs(y[jump_zone]))
    # az ugrástól távol eső kimeneti pozíciók
    others = np.delete(y, np.arange(max(0, JUMP + 1 - 2*K), min(len(y), JUMP + K)))
    return peak / np.std(others)

print(f"mozgóátlag (K=9) pontszáma:  {score_kernel(moving_avg):.2f}")
print(f"differencia [-1,0,1] pontszáma: {score_kernel(difference):.2f}")

In [ ]:
sajat_kernel = None  # ✍️ pl. [-1, 0, 1]
megallo(2, sajat_kernel)

y_sajat = conv1d_fixed(signal, sajat_kernel)
print(f"A kerneled pontszáma: {score_kernel(sajat_kernel):.2f}")
plt.figure(figsize=(12, 3))
plt.plot(y_sajat, color='tab:purple'); plt.axvline(JUMP - len(sajat_kernel)//2, color='gray', ls=':')
plt.title(f"A saját kerneled kimenete (K={len(sajat_kernel)})")

## 2. Az adatok

Letöltjük a jenai időjárási adatsort (kb. 13 MB). A 10 perces mintavétel túl sűrű a feladatunkhoz, ezért **óránként egy mintát** tartunk meg.

In [ ]:
zip_path = keras.utils.get_file(
    fname="jena_climate_2009_2016.csv.zip",
    origin="https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip")

df = pd.read_csv(zip_path)                      # a pandas a zip fájlt közvetlenül is be tudja olvasni
df = df[5::6]                                   # 10 perces -> óránkénti mintavétel
df.index = pd.to_datetime(df.pop('Date Time'), format='%d.%m.%Y %H:%M:%S')
print(df.shape)
df.head()

A bemeneti csatornák:
* 4 mért mennyiség: **hőmérséklet, légnyomás, relatív páratartalom, szélsebesség**,
* 2+2 „naptár” csatorna: a napszakot és az évszakot `sin`/`cos` párral kódoljuk. Így a 23:00 és a 0:00 egymáshoz közel esik, és a háló tudhatja, hogy éppen nappal vagy éjszaka, nyár vagy tél van.

A szélsebesség-adatokban van néhány hibás, -9999 értékű mérés, ezeket 0-ra cseréljük.

In [ ]:
df.loc[df['wv (m/s)'] < 0, 'wv (m/s)'] = 0.0      # hibás mérések javítása

hour = df.index.hour
doy  = df.index.dayofyear
features = pd.DataFrame({
    'T (degC)':  df['T (degC)'],
    'p (mbar)':  df['p (mbar)'],
    'rh (%)':    df['rh (%)'],
    'wv (m/s)':  df['wv (m/s)'],
    'nap sin':   np.sin(2*np.pi*hour/24),
    'nap cos':   np.cos(2*np.pi*hour/24),
    'ev sin':    np.sin(2*np.pi*doy/365.25),
    'ev cos':    np.cos(2*np.pi*doy/365.25),
}, index=df.index)
feature_names = list(features.columns)
features.describe().T[['mean', 'std', 'min', 'max']]

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7))
features['T (degC)'].plot(ax=axes[0], lw=0.5, title="Hőmérséklet, 2009–2016")
features['T (degC)'][:240].plot(ax=axes[1], marker='.', title="Az első 10 nap: jól látszik a napi ciklus")
for ax in axes: ax.set_ylabel("°C")
plt.tight_layout()

---
## 🗣️ Közös megálló 3 (≈ 25. perc): tippverseny 🏆

Nézd meg a fenti ábrákat! **Szerinted mekkora átlagos hibával (°C) lehet 24 órára előre jelezni a hőmérsékletet?**

* (a) a legegyszerűbb módszerrel: „24 óra múlva ugyanannyi fok lesz, mint most”,
* (b) egy kis 1D konvolúciós hálóval, amely az elmúlt 3 nap összes mérését látja.

Írd be a tippjeidet, és írd fel őket a táblára is! A gyakorlat során kiderül, kié volt a legközelebb.

In [ ]:
tipp_baseline_mae = None  # ✍️ °C, pl. 3.0
tipp_cnn_mae      = None  # ✍️ °C
megallo(3, tipp_baseline_mae, tipp_cnn_mae)

### Felosztás és standardizálás

Az adatokat időrendben osztjuk fel: a tanítóhalmaz a korai, a validációs a középső, a teszthalmaz a legkésőbbi időszak (70% / 15% / 15%). A standardizáláshoz az átlagot és a szórást **csak a tanítóhalmazból** számoljuk.

In [ ]:
data = features.values.astype('float32')
n = len(data)
train_end = int(n * 0.70)
valid_end = int(n * 0.85)

mean = data[:train_end].mean(axis=0)
std  = data[:train_end].std(axis=0)
data_std = (data - mean) / std

T_IDX = feature_names.index('T (degC)')
T_mean, T_std = mean[T_IDX], std[T_IDX]
print(f"tanító: {train_end}, validációs: {valid_end-train_end}, teszt: {n-valid_end} óra")

### Ablakozás

Minden tanítóminta egy **`window_size` órás bemeneti ablak** (az összes csatornával), a célérték pedig a hőmérséklet **`horizon` órával az ablak vége után**:

```
         bemenet: window_size óra                horizon óra
|----------------------------------------|....................|
t                                    t+window_size-1          t+window_size+horizon-1  <- célérték (T)
```

Az ablakokat minden halmazon külön képezzük, hogy egy ablak se lógjon át két halmaz határán. Az ablak 1 óránként lép tovább.

---
## 🗣️ Közös megálló 4 (≈ 33. perc): hány minta lesz?

1. A tanítóhalmaz hossza fent látható (órában). `window_size = 72` és `horizon = 24` mellett **hány tanító ablak (minta) keletkezik?** Írd be a tippedet!
2. Beszéljük meg: **miért nem keverjük meg** az adatokat a felosztás előtt, és **miért csak a tanítóhalmazból** számoljuk az átlagot és a szórást?

In [ ]:
tipp_tanito_ablakok = None  # ✍️ a tanító ablakok száma
megallo(4, tipp_tanito_ablakok)

In [ ]:
def make_timeseries_instances(data, target, window_size, horizon):
    X, Y = [], []
    for start in range(len(data) - window_size - horizon + 1):
        X.append(data[start:start + window_size])
        Y.append(target[start + window_size + horizon - 1])
    return np.array(X, dtype='float32'), np.array(Y, dtype='float32')

window_size = 72    # 3 nap visszatekintés
horizon     = 24    # 24 órás előrejelzés

splits = {'train': data_std[:train_end], 'valid': data_std[train_end:valid_end], 'test': data_std[valid_end:]}
X_train, Y_train = make_timeseries_instances(splits['train'], splits['train'][:, T_IDX], window_size, horizon)
X_valid, Y_valid = make_timeseries_instances(splits['valid'], splits['valid'][:, T_IDX], window_size, horizon)
X_test,  Y_test  = make_timeseries_instances(splits['test'],  splits['test'][:, T_IDX],  window_size, horizon)

print("X_train:", X_train.shape, " (minták, időlépések, csatornák)")
print("Y_train:", Y_train.shape)
print(f"tanító ablakok száma: {len(X_train)}   (tipped: {tipp_tanito_ablakok})")

✍️ **Válasz (4. megálló, 2. kérdés):** Miért nem keverjük meg az adatokat a felosztás előtt? Miért csak a tanítóhalmazból számolunk átlagot és szórást?

*(ide írd a válaszod)*

## 3. Baseline: „holnap ugyanannyi lesz, mint most”

A legegyszerűbb előrejelzés: a 24 óra múlva várható hőmérséklet **egyezik a mostanival** (az ablak utolsó értékével).

A hibát **átlagos abszolút hibában (MAE), °C-ban** mérjük, ehhez visszaskálázzuk az értékeket.

In [ ]:
def to_celsius(y_std):
    return y_std * T_std + T_mean

baseline_preds = X_test[:, -1, T_IDX]            # az ablak utolsó hőmérséklet-értéke
baseline_mae = np.mean(np.abs(to_celsius(baseline_preds) - to_celsius(Y_test)))
print(f"Baseline MAE a teszthalmazon: {baseline_mae:.2f} °C   (tipped: {tipp_baseline_mae} °C)")

## 4. Az 1D konvolúciós hálózat

A hálózat felépítése:
* két `Conv1D` + `MaxPooling1D` blokk: a konvolúciók rövid időbeli mintázatokat (pl. melegedés, hidegfront, a napi ciklus egy szakasza) keresnek, a pooling felezi az időbeli felbontást,
* `Flatten` + `Dense(1)`: a megtalált mintázatokból és azok **időbeli helyéből** egyetlen számot (a jövőbeli hőmérsékletet) számol, lineáris kimenettel, hiszen ez regresszió.

Hasznos képletek:
* egy `Conv1D` réteg paraméterszáma: $(K \cdot C_{be} + 1)\cdot C_{ki}$, ahol $K$ a kernelméret, $C_{be}$ a bemeneti, $C_{ki}$ a kimeneti csatornák (szűrők) száma,
* `padding='valid'` esetén a kimenet hossza $L - K + 1$, a `MaxPooling1D(2)` után $\lfloor L/2 \rfloor$.

**Receptív mező:** a második pooling után egy pozíció a bemenet 16 egymást követő órájára „lát rá” ($5 \to 6 \to 6+4\cdot2=14 \to 16$).

In [ ]:
def make_1d_convnet(window_size, nb_channels, filter_length=5, nb_filter=16):
    model = keras.Sequential([
        keras.Input(shape=(window_size, nb_channels)),
        layers.Conv1D(filters=nb_filter, kernel_size=filter_length, activation='relu'),
        layers.MaxPooling1D(pool_size=2),
        layers.Conv1D(filters=2*nb_filter, kernel_size=filter_length, activation='relu'),
        layers.MaxPooling1D(pool_size=2),
        layers.Flatten(),
        layers.Dense(1, activation='linear'),
    ])
    model.compile(loss='mse', optimizer=keras.optimizers.Adam(learning_rate=1e-3), metrics=['mae'])
    return model

---
## 🗣️ Közös megálló 5 (≈ 42. perc): számoljatok párban!

A fenti `make_1d_convnet` függvényt `window_size=72`, 8 bemeneti csatorna, `filter_length=5`, `nb_filter=16` értékekkel hívjuk meg. **Párban, papíron** számoljátok ki:
1. az első `Conv1D` réteg paraméterszámát,
2. a második `Conv1D` réteg paraméterszámát (figyelem: ennek `2*nb_filter` szűrője van!),
3. a `Flatten` réteg kimenetének méretét.

A következő cella a `summary()` alapján ellenőrzi a tippjeiteket.

In [ ]:
tipp_param_conv1 = None  # ✍️
tipp_param_conv2 = None  # ✍️
tipp_flatten     = None  # ✍️
megallo(5, tipp_param_conv1, tipp_param_conv2, tipp_flatten)

In [ ]:
keras.utils.set_random_seed(42)   # így az eredmény nem függ attól, hány kernelt próbáltál ki a versenyben
model = make_1d_convnet(window_size, nb_channels=X_train.shape[-1])
model.summary()

valodi = [model.layers[0].count_params(), model.layers[2].count_params(), model.layers[-1].kernel.shape[0]]
tippek = [tipp_param_conv1, tipp_param_conv2, tipp_flatten]
for nev, tipp, v in zip(['1. Conv1D paraméterei', '2. Conv1D paraméterei', 'Flatten kimenet mérete'], tippek, valodi):
    print(f"{nev:25s} tipped: {tipp},  valódi: {v}  {'✅' if tipp == v else '❌'}")

Tanítás. Az `EarlyStopping` leállítja a tanítást, ha a validációs hiba 5 epoch óta nem javul, és visszaállítja a legjobb súlyokat.

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(X_train, Y_train,
                    epochs=30, batch_size=64,
                    validation_data=(X_valid, Y_valid),
                    callbacks=[early_stopping], verbose=2)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(np.array(history.history['mae']) * T_std, label='tanító MAE')
plt.plot(np.array(history.history['val_mae']) * T_std, label='validációs MAE')
plt.axhline(baseline_mae, color='gray', ls='--', label='baseline (teszt)')
plt.xlabel('epoch'); plt.ylabel('MAE [°C]'); plt.legend(); plt.grid(alpha=0.3)

### Kiértékelés a teszthalmazon

In [ ]:
preds = model.predict(X_test, verbose=0).ravel()
cnn_mae = np.mean(np.abs(to_celsius(preds) - to_celsius(Y_test)))
print(f"Baseline MAE: {baseline_mae:.2f} °C   (tipped: {tipp_baseline_mae} °C)")
print(f"1D CNN MAE:   {cnn_mae:.2f} °C   (tipped: {tipp_cnn_mae} °C)")

In [ ]:
# két hét a teszthalmazból
sl = slice(0, 24*14)
plt.figure(figsize=(14, 5))
plt.plot(to_celsius(Y_test[sl]), color='g', lw=2, label='valódi hőmérséklet')
plt.plot(to_celsius(baseline_preds[sl]), color='gray', ls='--', label='baseline (24 órával korábbi érték)')
plt.plot(to_celsius(preds[sl]), color='r', label='1D CNN előrejelzés')
plt.xlabel('óra'); plt.ylabel('°C'); plt.title('24 órás előrejelzés, két hét a teszthalmazból')
plt.legend(); plt.grid(alpha=0.3)

---
## 🗣️ Közös megálló 6 (≈ 52. perc): siker ez?

1. Kinek a tippje volt a legközelebb a 3. megállóban? 🏆
2. **Siker-e** a CNN eredménye a baseline-hoz képest? Mitől erős a baseline?
3. A fenti ábrán: hol téved nagyot a baseline, és hol a CNN? Mi látszik a baseline görbéjén?

✍️ **Válasz (6. megálló):**

*(ide írd a válaszod)*

In [ ]:
megallo(6)

## 5. Mit tanultak az első réteg szűrői?

Az első `Conv1D` réteg súlyainak alakja `(kernel_size, bemeneti csatornák, szűrők) = (5, 8, 16)`. Minden szűrő tehát egy kis 5 × 8-as „minta”: megmutatja, hogy a szűrő melyik csatorna melyik időpillanatát mekkora súllyal veszi figyelembe (piros: pozitív, kék: negatív súly).

In [ ]:
w = model.layers[0].get_weights()[0]      # (kernel_size, csatornák, szűrők)
print("Első réteg súlyainak alakja:", w.shape)

n_show = 6
vmax = np.abs(w).max()
fig, axes = plt.subplots(1, n_show, figsize=(16, 4), sharey=True)
for i, ax in enumerate(axes):
    ax.imshow(w[:, :, i].T, cmap='coolwarm', vmin=-vmax, vmax=vmax, aspect='auto')
    ax.set_title(f"szűrő {i}")
    ax.set_xlabel("kernel pozíció")
axes[0].set_yticks(range(len(feature_names)))
axes[0].set_yticklabels(feature_names)
plt.tight_layout()

---
## 🗣️ Közös megálló 7 (≈ 58. perc): szűrő-sztorik

Válasszatok csoportonként **egy szűrőt** a fenti ábráról, és találjátok ki, mit érzékelhet! Például: „ha az utolsó órákban nő a hőmérséklet és közben csökken a légnyomás, akkor nagy a kimenete”.

Vannak-e a bemelegítésben látott szűrőkre emlékeztető minták? (Egy csatornán belül előjelet váltó súlyok: *változást* érzékelő szűrő; azonos előjelű súlyok: *átlagoló* szűrő.)

✍️ **Válasz (7. megálló):** a választott szűrő száma, és szerintünk mit érzékel:

*(ide írd a válaszod)*

In [ ]:
megallo(7)

## 6. Feladatok és ranglista-verseny

Minden feladatnál három dolgot találsz:
* egy **kódcellát** a kísérletekhez: módosítsd a paramétereket, és futtasd újra,
* egy **táblázatot**, amelybe a mért eredményeidet írod,
* egy **válaszcellát** (✍️), amelybe a kérdésekre adott válaszodat írod.

**Fontos:** a kísérleteknél a beállítások közötti választáshoz a **validációs hibát** használjuk. A segédfüggvény a teszthibát elrejti, és csak a verseny végén nézzük meg (9. megálló).

A kísérletekhez az alábbi segédfüggvényt használd. Ez a fenti lépéseket egyetlen hívásba foglalja, és minden futtatás eredményét elmenti az `eredmenyek` listába. Egy futtatás CPU-n kb. egy perc.

In [ ]:
eredmenyek = []   # minden kísérlet eredménye ide kerül

def train_and_evaluate(window_size=72, horizon=24, filter_length=5, nb_filter=16,
                       channels=None, build_fn=None, nev=None, epochs=30, show_test=False, verbose=0):
    # channels: a bemeneti csatornák indexeinek listája (None = mind a 8)
    # build_fn: saját modellépítő függvény build_fn(window_size, nb_channels) alakban (None = make_1d_convnet)
    # nev: a kísérlet neve a ranglistán
    if channels is None:
        channels = list(range(len(feature_names)))
    X_tr, Y_tr = make_timeseries_instances(splits['train'], splits['train'][:, T_IDX], window_size, horizon)
    X_va, Y_va = make_timeseries_instances(splits['valid'], splits['valid'][:, T_IDX], window_size, horizon)
    X_te, Y_te = make_timeseries_instances(splits['test'],  splits['test'][:, T_IDX],  window_size, horizon)
    base_va = np.mean(np.abs(X_va[:, -1, T_IDX] - Y_va)) * T_std
    base_te = np.mean(np.abs(X_te[:, -1, T_IDX] - Y_te)) * T_std
    X_tr, X_va, X_te = X_tr[:, :, channels], X_va[:, :, channels], X_te[:, :, channels]

    keras.utils.set_random_seed(42)
    if build_fn is None:
        m = make_1d_convnet(window_size, len(channels), filter_length=filter_length, nb_filter=nb_filter)
    else:
        m = build_fn(window_size, len(channels))
    m.compile(loss='mse', optimizer=keras.optimizers.Adam(learning_rate=1e-3), metrics=['mae'])
    es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    h = m.fit(X_tr, Y_tr, epochs=epochs, batch_size=64, validation_data=(X_va, Y_va),
              callbacks=[es], verbose=verbose)
    cnn_va = np.mean(np.abs(m.predict(X_va, verbose=0).ravel() - Y_va)) * T_std
    cnn_te = np.mean(np.abs(m.predict(X_te, verbose=0).ravel() - Y_te)) * T_std

    if nev is None:
        nev = (f"w={window_size}, k={filter_length}, f={nb_filter}, csat={len(channels)}"
               + ("" if build_fn is None else f", {getattr(build_fn, '__name__', 'saját')}"))
    eredmenyek.append({'kísérlet': nev, 'horizon': horizon, 'paraméterek': m.count_params(),
                       'valid baseline MAE': round(base_va, 3), 'valid MAE': round(cnn_va, 3),
                       'teszt baseline MAE': round(base_te, 3), 'teszt MAE': round(cnn_te, 3),
                       'epochok': len(h.history['loss'])})
    print(f"{nev}, horizon={horizon} | paraméterek: {m.count_params()} | "
          f"VALID baseline MAE: {base_va:.2f} °C | VALID CNN MAE: {cnn_va:.2f} °C | epochok: {len(h.history['loss'])}")
    if show_test:
        print(f"   TESZT baseline MAE: {base_te:.2f} °C | TESZT CNN MAE: {cnn_te:.2f} °C")
    return m

---
## 🗣️ Közös megálló 8 (≈ 60. perc): csoportbeosztás és a verseny szabályai

* Minden csoport megkapja az **1–4. feladat** közül az egyiket. **10 perc** kísérletezés után minden csoport **2 percben beszámol**, és a közös táblázatba beírja az eredményeit.
* Az 5. feladat szorgalmi (vagy otthoni) feladat.
* **Ranglista-verseny 🏆:** közben mindenki próbáljon minél kisebb **validációs** hibát elérni `horizon=24` mellett, tetszőleges beállítással (ablakméret, kernelméret, szűrőszám, csatornák, saját architektúra). A legjobb eredményeteket írjátok fel a táblára!

In [ ]:
megallo(8)

### 1. feladat: ablakméret és kernelméret

Próbáld ki a `window_size` = 24, 120 és a `filter_length` = 3, 9, 15 értékeket (egyszerre csak az egyiket változtasd, a másik maradjon az alapérték). Hogyan változik a paraméterszám, a receptív mező és a hiba?

*Tipp a receptív mezőhöz:* egy $K$ méretű konvolúció $(K-1)\cdot j$-vel, egy 2-es pooling $1\cdot j$-vel növeli a receptív mezőt, ahol $j$ az adott réteg előtti összesített lépésköz (a pooling után duplázódik). Az alapmodellnél: $5 \to 6 \to 14 \to 16$.

In [ ]:
# Módosítsd az értékeket, és futtasd újra a cellát minden beállításra!
model_1 = train_and_evaluate(window_size=24, filter_length=5)

| `window_size` | `filter_length` | paraméterek száma | receptív mező (óra) | valid MAE [°C] |
|---|---|---|---|---|
| 72 | 5 | 3729 | 16 | |
| 24 | 5 | | | |
| 120 | 5 | | | |
| 72 | 3 | | | |
| 72 | 9 | | | |
| 72 | 15 | | | |

✍️ **Válasz:** Mindig jobb, ha a háló hosszabb múltat lát? Mivel magyarázod az eredményt? Hogyan függ a paraméterszám a `window_size`-tól, és miért?

*(ide írd a válaszod)*

### 2. feladat: bemeneti csatornák

Tanítsd a hálót csak a hőmérséklet-csatornával (`channels=[T_IDX]`), majd egy általad választott csatorna-részhalmazzal. Mennyit segít a többi mért mennyiség és a naptár-kódolás?

In [ ]:
print(list(enumerate(feature_names)))
model_2 = train_and_evaluate(channels=[T_IDX])
# saját választás, pl.:
# model_2 = train_and_evaluate(channels=[feature_names.index(c) for c in ['T (degC)', 'nap sin', 'nap cos']])

| bemeneti csatornák | csatornák száma | az első Conv1D paraméterei | valid MAE [°C] |
|---|---|---|---|
| mind | 8 | | |
| csak `T (degC)` | 1 | | |
| saját: … | | | |

✍️ **Válasz:** Mennyit javítanak a további csatornák? Szerinted melyik a legfontosabb a hőmérsékleten kívül, és miért? Hogyan változott az első konvolúciós réteg paraméterszáma, és miért pont így?

*(ide írd a válaszod)*

### 3. feladat: előrejelzési horizont

Állítsd a `horizon` értékét 1, 6, 24 és 48 órára.

In [ ]:
model_3 = train_and_evaluate(horizon=1)

| `horizon` (óra) | valid baseline MAE [°C] | valid CNN MAE [°C] | javulás a baseline-hoz képest (%) |
|---|---|---|---|
| 1 | | | |
| 6 | | | |
| 24 | | | |
| 48 | | | |

✍️ **Válasz:** Melyik horizontnál veri meg leginkább a CNN a baseline-t? Miért éppen ott gyenge a baseline? (Tipp: gondolj a napi ciklusra.)

*(ide írd a válaszod)*

### 4. feladat: dilatált konvolúció

A `dilation_rate=d` paraméterrel a kernel elemei között $d-1$ kihagyott időlépés van, így a kernel nagyobb időtartamot fed le. Próbáld ki a `dilation_rate` = 2 és 4 értékeket!

In [ ]:
def make_dilated_convnet(window_size, nb_channels, filter_length=5, nb_filter=16, dilation_rate=2):
    return keras.Sequential([
        keras.Input(shape=(window_size, nb_channels)),
        layers.Conv1D(nb_filter, filter_length, dilation_rate=dilation_rate, activation='relu'),
        layers.MaxPooling1D(pool_size=2),
        layers.Conv1D(2*nb_filter, filter_length, dilation_rate=dilation_rate, activation='relu'),
        layers.MaxPooling1D(pool_size=2),
        layers.Flatten(),
        layers.Dense(1, activation='linear'),
    ])

d = 2
model_4 = train_and_evaluate(build_fn=lambda w, c: make_dilated_convnet(w, c, dilation_rate=d), nev=f"dilatált, d={d}")
model_4.summary()

| `dilation_rate` | a Conv1D rétegek paraméterei | `Flatten` kimenet mérete | receptív mező (óra, számolt) | valid MAE [°C] |
|---|---|---|---|---|
| 1 | | | 16 | |
| 2 | | | | |
| 4 | | | | |

✍️ **Válasz:** Hogyan számoltad ki a receptív mezőt? Változott-e a konvolúciós rétegek paraméterszáma, és miért? Miért változott mégis a teljes paraméterszám?

*(ide írd a válaszod)*

### 5. feladat (szorgalmi): Flatten vs. GlobalAveragePooling1D

A lenti modell most még az alapmodellel egyezik. Cseréld le benne a `Flatten` réteget `GlobalAveragePooling1D`-re, és hasonlítsd össze a két változatot!

In [ ]:
def make_my_convnet(window_size, nb_channels, filter_length=5, nb_filter=16):
    return keras.Sequential([
        keras.Input(shape=(window_size, nb_channels)),
        layers.Conv1D(nb_filter, filter_length, activation='relu'),
        layers.MaxPooling1D(pool_size=2),
        layers.Conv1D(2*nb_filter, filter_length, activation='relu'),
        layers.MaxPooling1D(pool_size=2),
        layers.Flatten(),                 # TODO: cseréld layers.GlobalAveragePooling1D()-re
        layers.Dense(1, activation='linear'),
    ])

model_5 = train_and_evaluate(build_fn=make_my_convnet)
model_5.summary()

| a konvolúciók utáni réteg | a `Dense` bemenetének mérete | a `Dense` paraméterei | valid MAE [°C] |
|---|---|---|---|
| `Flatten` | | | |
| `GlobalAveragePooling1D` | | | |

✍️ **Válasz:** Hogyan változott a paraméterszám és a hiba? Mi az az információ, ami a `GlobalAveragePooling1D` használatakor elvész? (Tipp: *hol* volt az ablakban egy minta?) Milyen feladatnál lenne ez inkább előny?

*(ide írd a válaszod)*

### 🏆 Ranglista

Az összes eddigi kísérleted a validációs hiba szerint rendezve (`horizon=24`):

In [ ]:
ranglista = pd.DataFrame(eredmenyek)
ranglista = ranglista[ranglista['horizon'] == 24].sort_values('valid MAE')
ranglista[['kísérlet', 'paraméterek', 'valid baseline MAE', 'valid MAE', 'epochok']]

---
## 🗣️ Közös megálló 9 (≈ 80. perc): eredményhirdetés

1. Kinek lett a legkisebb **validációs** hibája? 🏆
2. Miért a validációs és nem a teszthiba alapján hirdetünk eredményt? Mi történne, ha mindenki a teszthibát nézve választaná ki a legjobb beállítását?
3. A megálló után nézzük meg a teszthibát is: ugyanaz a beállítás nyer?

✍️ **Válasz (9. megálló):**

*(ide írd a válaszod)*

In [ ]:
megallo(9)

In [ ]:
# Csak a verseny lezárása után! A teljes táblázat a teszthibával együtt:
ranglista = pd.DataFrame(eredmenyek)
ranglista = ranglista[ranglista['horizon'] == 24].sort_values('valid MAE')
display(ranglista)
if len(ranglista) > 0:
    print("Validáció szerint legjobb:", ranglista.iloc[0]['kísérlet'])
    print("Teszt szerint legjobb:    ", ranglista.sort_values('teszt MAE').iloc[0]['kísérlet'])

---
## 🗣️ Közös megálló 10 (≈ 85. perc): összegzés

Egy kör: mindenki egy mondatban elmondja, **mi volt a legmeglepőbb** a mai gyakorlaton. Írd be a lenti cellába is!

✍️ **Válasz:** Foglald össze 3–5 mondatban, mit tanultál az 1D konvolúcióról ebből a notebookból!

*(ide írd a válaszod)*

In [ ]:
legmeglepobb = None  # ✍️ egy mondat, idézőjelek között
megallo(10, legmeglepobb)